In [1]:
%pip install pyarrow

   ---------------------------------------- 0.0/27.6 MB ? eta -:--:--
   --- ------------------------------------ 2.4/27.6 MB 19.2 MB/s eta 0:00:02
   ---------------- ----------------------- 11.3/27.6 MB 35.2 MB/s eta 0:00:01
   ----------------------------- ---------- 20.7/27.6 MB 39.6 MB/s eta 0:00:01
   ---------------------------------------- 27.6/27.6 MB 38.1 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [27]:
from pathlib import Path
import pandas as pd
import re

**Data exploration**

In [3]:
REPO_ROOT = Path(".").resolve().parent
DATA_PATH = REPO_ROOT / "data"
MODEL_PATH = REPO_ROOT / "models"
OUTPUT_PATH = REPO_ROOT / "output"
INDEX_PATH = REPO_ROOT / "data" / "processed"

LAWS_CSV = DATA_PATH / "laws_de.csv"
COURTS_CSV = DATA_PATH / "court_considerations.csv"
TRAIN_CSV = DATA_PATH / "train.csv"
TEST_CSV = DATA_PATH / "test.csv"
VAL_CSV = DATA_PATH / "val.csv"


In [4]:
train_data = pd.read_csv(TRAIN_CSV)

train_data.head(5)

,query_id,query,gold_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...


In [5]:
laws_data = pd.read_csv(LAWS_CSV)   

laws_data.head(5)

,citation,text,title
0,Art. 1 112,Die Einwohnergemeinde Bern tritt der Schweizer...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
1,Art. 2 112,Die Einwohnergemeinde Bern wird ferner der Sch...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
2,Art. 3 Abs. 1 112,1 Falls die Schweizerische Eidgenossenschaft z...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
3,Art. 3 Abs. 2 112,2 Durch Anlage des neuen Verwaltungsgebäudes a...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...
4,Art. 4 Abs. 1 112,1 Die Einwohnergemeinde Bern übernimmt im fern...,Übereinkunft vom 22. Juni 1875 zwischen dem Sc...


In [6]:
court_csv = pd.read_csv(COURTS_CSV)   

court_csv.head(5)

,citation,text
0,BGE 139 I 2 E. 1.12.2011,betr. Verweigerung der Beiladung seien gutzuhe...
1,BGE 139 I 2 E. 2,Eventualiter sei die Rückweisung an die Vorins...
2,BGE 139 I 2 E. 5.1,"In der Sache ist vorweg zu prüfen, ob der Ents..."
3,BGE 139 I 2 E. 5.2,Art. 34 Abs. 1 BV gewährleistet in allgemeiner...
4,BGE 139 I 2 E. 5.3,Im vorliegenden Fall geht es nicht um die Gült...


In [7]:
print(f"Total Rows: {len(train_data)}")
print(train_data.isnull().sum())

Total Rows: 1139
query_id          0
query             0
gold_citations    0
dtype: int64


In [8]:
train_data['query_len'] = train_data['query'].apply(len)
train_data['num_citations'] = train_data['gold_citations'].apply(lambda x: len(str(x).split(';')))

print(train_data[['query_len', 'num_citations']].describe())

          query_len  num_citations
count   1139.000000    1139.000000
mean    1620.304653       4.090430
std     3019.878360       4.512206
min       40.000000       1.000000
25%      441.000000       1.000000
50%     1016.000000       2.000000
75%     1795.000000       5.000000
max    32767.000000      44.000000


In [9]:
train_data.head(5)

,query_id,query,gold_citations,query_len,num_citations
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...,1495,3
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...,1376,11
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB,1095,1
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....,2899,4
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...,5763,6


In [10]:
all_citations = train_data['gold_citations'].str.split(';').explode().str.strip()

citation_counts = all_citations.value_counts().reset_index()
citation_counts.describe()

,count
count,2695.000000
mean,1.728757
std,1.505338
min,1.000000
25%,1.000000
50%,1.000000
75%,2.000000
max,20.000000


In [11]:
list(all_citations)

['Art. 10a Abs. 1 USG',
 'Art. 2 Abs. 1 UVPV',
 'Art. 10a Abs. 1 UVG',
 'Art. 975 ZGB',
 'Art. 974 Abs. 2 ZGB',
 'Art. 973 ZGB',
 'Art. 661 ZGB',
 'Art. 956a ZGB',
 'Art. 956a Abs. 3 ZGB',
 'Art. 955 Abs. 1 ZGB',
 'Art. 976 ZGB',
 'Art. 976a ZGB',
 'Art. 60 OR',
 'Art. 955 Abs. 2 ZGB',
 'Art. 264m StGB',
 'Art. 176 Abs. 1 IPRG',
 'Art. 186 Abs. 1 IPRG',
 'Art. 186 Abs. 3 IPRG',
 'Art. 190 Abs. 3 IPRG',
 'Art. 93 Abs. 1 BGG',
 'Art. 89 Abs. 1 BGG',
 'Art. 89 Abs. 2 BGG',
 'Art. 50 BV',
 'Art. 95 BGG',
 'Art. 106 Abs. 2 BGG',
 'Art. 42 Abs. 1 StGB',
 'Art. 106 StGB',
 'Art. 42 Abs. 4 StGB',
 'Art. 626 Abs. 1 ZGB',
 'Art. 626 Abs. 2 ZGB',
 'Art. 630 ZGB',
 'Art. 69 DBG',
 'Art. 358 OR',
 'Art. 337 OR',
 'Art. 361 OR',
 'Art. 11 StGB',
 'Art. 29 StGB',
 'Art. 111 StGB',
 'Art. 158 StGB',
 'Art. 117 StGB',
 'Art. 102 Abs. 1 StGB',
 'Art. 102 Abs. 2 StGB',
 'Art. 102 StGB',
 'Art. 102 Abs. 4 StGB',
 'Art. 706 OR',
 'Art. 717 OR',
 'Art. 643 OR',
 'Art. 716a Abs. 1 OR',
 'Art. 12 Abs. 2 StGB'

**Check data against provided corpus**

In [ ]:
train_citations_set = set(all_citations.unique())

articles_in_laws = set(laws_data['citation'].values)
articles_in_courts = set(court_csv['citation'].values)

In [ ]:
intersect_laws = train_citations_set.intersection(articles_in_laws)
non_intersect_laws = train_citations_set.difference(articles_in_laws)
intersect_courts = train_citations_set.intersection(articles_in_courts)
non_intersect_courts = train_citations_set.difference(articles_in_courts)

In [14]:
print(f"In laws: {len(articles_in_laws)}")
print(f"In courts: {len(articles_in_courts)}")
print(f"In training data intersection with laws: {len(intersect_laws)}")
print(f"In training data intersection with courts: {len(intersect_courts)}")
print(f"Not in laws: {len(non_intersect_laws)}")
print(f"Not in courts: {len(non_intersect_courts)}")

In laws: 175933
In courts: 1985178
In training data intersection with laws: 1876
In training data intersection with courts: 52
Not in laws: 819
Not in courts: 2643


There are some cases in training where the articles are not in the corpus and also not in val set - consider dropping them.

In [16]:
in_training_not_in_laws_not_in_courts = train_citations_set.difference(articles_in_laws.union(articles_in_courts))
print(f"In training data but not in laws or courts: {len(in_training_not_in_laws_not_in_courts)}")
print(",".join(list(in_training_not_in_laws_not_in_courts)))

In training data but not in laws or courts: 767
Art. 217 StPO,Art. 24a RPG,Art. 395 ZGB,Art. 522 ZGB,Art. 181 DBG,Art. 3 Abs. 3 StPO,Art. 549 ZGB,Art. 68 AIG,Art. 652 OR,Art. 659 OR,Art. 23 VStG,Art. 282 StGB,Art. 660 OR,Art. 25 URG,Art. 28 OR,Art. 27 ArG,Art. 712c ZGB,Art. 195a ZGB,Art. 28a ZGB,Art. 24 BV,Art. 476 ZGB,Art. 190 StGB,Art. 47 EleG,Art. 31c USG,Art. 5 GBV,Art. 13 JStG,Art. 18k EBG,Art. 41 OR,Art. 58 FIDLEG,Art. 3 DSG,Art. 61 Abs. 1 URG,Art. 18a DSG,Art. 257 ZPO,Art. 108 AsylG,Art. 5 Abs. 1 FIDLEG,Art. 28 JStG,Art. 706 OR,Art. 264m StGB,Art. 470 ZGB,Art. 12 ArG,Art. 23 JStG,Art. 209 ZGB,Art. 732 ZGB,Art. 138 StGB,Art. 15 Abs. 1 HKsÜ,Art. 96 IPRG,Art. 31 OR,Art. 419 Abs. 1 OR,Art. 15 JStG,Art. 9 UVG,Art. 100 SVG,Art. 15 RPG,Art. 66a StGB,Art. 177 DBG,Art. 83 GBV,Art. 34 Abs. 2 RPV,Art. 9 RPG,Art. 21 IPRG,Art. 11a Abs. 2 DSG,Art. 36 BV,Art. 237 ZPO,Art. 34 DSG,Art. 260 StPO,Art. 493 ZGB,Art. 3 URG,Art. 8 AsylG,Art. 186 DBG,Art. 24b RPG,Art. 9 JStPO,Art. 647d ZGB,Art. 210 OR,

In [17]:
val_data = pd.read_csv(VAL_CSV)
val_data.head(5)

,query_id,query,gold_citations
0,val_001,May a court lawfully order a three‑month exten...,Art. 221 Abs. 1 StPO;Art. 140 Abs. 1 StGB;Art....
1,val_002,A claimant holding a national vocational diplo...,Art. 8 Abs. 1 ATSG;Art. 8 Abs. 1 IVG;Art. 17 A...
2,val_003,"A. Rivera, a Peruvian national born in 1994 an...",Art. 29 Abs. 2 BV;Art. 221 Abs. 1 StPO;Art. 39...
3,val_004,"Mr. Dalton, born in 1941 and resident in a sma...",Art. 505 Abs. 1 ZGB;Art. 467 ZGB;Art. 469 Abs....
4,val_005,"A parent, separated from their co-parent since...",Art. 133 Abs. 1 ZGB;Art. 133 Abs. 2 ZGB;Art. 2...


In [18]:
test_data = pd.read_csv(TEST_CSV)
test_data.head(5)

,query_id,query
0,test_001,Four U.S.-based software companies (NorthWave ...
1,test_002,On 9 August 2011 a 62‑year‑old cyclist (the cl...
2,test_003,"On 12 March 2012, Meridian Leasing Ltd and Ori..."
3,test_004,"A publicly listed manufacturing company, Orion..."
4,test_005,"A logistics company (R Ltd.), which owns a dis..."


In [19]:
val_not_in_laws_not_in_courts = set(val_data['gold_citations'].str.split(';').explode().str.strip().unique()).difference(articles_in_laws.union(articles_in_courts))
print(f"In validation data but not in laws or courts: {len(val_not_in_laws_not_in_courts)}")

In validation data but not in laws or courts: 0


In [20]:
val_in_training_not_in_laws_not_in_courts = set(val_data['gold_citations'].str.split(';').explode().str.strip().unique()).intersection(in_training_not_in_laws_not_in_courts)
print(f"In validation data and in training data but not in laws or courts: {len(val_in_training_not_in_laws_not_in_courts)}")

In validation data and in training data but not in laws or courts: 0


In [21]:
in_training_not_in_laws_not_in_courts

{'Art. 1 Abs. 1 LugÜ',
 'Art. 1 Abs. 2 LugÜ',
 'Art. 1 LugÜ',
 'Art. 1 MSchG',
 'Art. 1 OR',
 'Art. 1 RPG',
 'Art. 1 VBVV',
 'Art. 10 ArG',
 'Art. 10 BV',
 'Art. 10 HKsÜ',
 'Art. 10 JStPO',
 'Art. 10 URG',
 'Art. 10 UVG',
 'Art. 100 AIG',
 'Art. 100 BGG',
 'Art. 100 SVG',
 'Art. 101 KVV',
 'Art. 101 OR',
 'Art. 102 StGB',
 'Art. 1028 OR',
 'Art. 103 BGG',
 'Art. 105 BGG',
 'Art. 106 StGB',
 'Art. 107a AsylG',
 'Art. 108 AsylG',
 'Art. 108 SchKG',
 'Art. 1085 OR',
 'Art. 10a Abs. 1 UVG',
 'Art. 10a USG',
 'Art. 11 OR',
 'Art. 11 PrHG',
 'Art. 11 StGB',
 'Art. 11 TEVG',
 'Art. 11 URG',
 'Art. 11 VVEA',
 'Art. 111 BGG',
 'Art. 112 IPRG',
 'Art. 112 OR',
 'Art. 113 StPO',
 'Art. 116 IPRG',
 'Art. 116 OR',
 'Art. 118 OR',
 'Art. 118 StPO',
 'Art. 118 ZPO',
 'Art. 119 BGG',
 'Art. 119 StGB',
 'Art. 11a Abs. 2 DSG',
 'Art. 11a Abs. 3 DSG',
 'Art. 12 ArG',
 'Art. 12 BüG',
 'Art. 12 DBG',
 'Art. 12 DSG',
 'Art. 12 JStG',
 'Art. 12 JStPO',
 'Art. 12 NHG',
 'Art. 12 SVKG',
 'Art. 12 SebG',
 'Art.

But they are found in "text" column in court data?

In [26]:
found = set()

for candidate in in_training_not_in_laws_not_in_courts:
    with open(COURTS_CSV, 'r', encoding="utf-8") as f:
        for line in f:
            if candidate in line:
                print(f"Candidatr {candidate} found in courts data.")
                found.add(candidate)
                break

Candidatr Art. 217 StPO found in courts data.
Candidatr Art. 24a RPG found in courts data.
Candidatr Art. 395 ZGB found in courts data.
Candidatr Art. 522 ZGB found in courts data.
Candidatr Art. 181 DBG found in courts data.
Candidatr Art. 68 AIG found in courts data.
Candidatr Art. 652 OR found in courts data.
Candidatr Art. 659 OR found in courts data.
Candidatr Art. 23 VStG found in courts data.
Candidatr Art. 282 StGB found in courts data.
Candidatr Art. 660 OR found in courts data.
Candidatr Art. 25 URG found in courts data.
Candidatr Art. 28 OR found in courts data.
Candidatr Art. 27 ArG found in courts data.
Candidatr Art. 712c ZGB found in courts data.
Candidatr Art. 195a ZGB found in courts data.
Candidatr Art. 28a ZGB found in courts data.
Candidatr Art. 24 BV found in courts data.
Candidatr Art. 476 ZGB found in courts data.
Candidatr Art. 190 StGB found in courts data.
Candidatr Art. 47 EleG found in courts data.
Candidatr Art. 31c USG found in courts data.
Candidatr Art. 

KeyboardInterrupt: 

In [28]:
def get_base_article(citation):
    base = re.sub(r'\sAbs\.\s\d+', '', citation)
    base = re.sub(r'\sZiff\.\s\d+', '', base)
    return base.strip()

In [29]:
from collections import defaultdict
parent_to_children = defaultdict(list)

for cit in articles_in_laws:
    parent = get_base_article(cit)
    parent_to_children[parent].append(cit)

In [34]:
true_for_cleaning = set()

for candidate_to_delete in in_training_not_in_laws_not_in_courts:
    candidate_parent = get_base_article(candidate_to_delete)
    if candidate_parent not in parent_to_children:
        true_for_cleaning.add(candidate_to_delete)

print(f"Total candidates in training data not in laws or courts: {len(in_training_not_in_laws_not_in_courts)}")
print(f"Total articles for cleaning: {len(true_for_cleaning)}")

Total candidates in training data not in laws or courts: 767
Total articles for cleaning: 53


In [35]:
valid_ids = set(articles_in_laws).union(set(articles_in_courts))
valid_ids = valid_ids.union(set(parent_to_children.keys()))

def clean_gold_labels(label_string):
    labels = [l.strip() for l in label_string.split(';')]
    
    valid_labels = [l for l in labels if l in valid_ids]

    return ";".join(valid_labels)


# use a copy just in care we need the original later, but we want to add a new column with cleaned gold labels

copy_train_data = train_data.copy()
copy_train_data['cleaned_gold'] = copy_train_data['gold_citations'].apply(clean_gold_labels)

copy_train_data = copy_train_data[copy_train_data['cleaned_gold'] != ""].copy()
copy_train_data.head(5)

,query_id,query,gold_citations,query_len,num_citations,cleaned_gold
0,train_0001,Die A AG betreibt seit den 1970er-Jahren auf d...,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV;Art. 10...,1495,3,Art. 10a Abs. 1 USG;Art. 2 Abs. 1 UVPV
1,train_0002,Die Alpha Consulting AG kann nun ihr Grundstüc...,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...,1376,11,Art. 975 ZGB;Art. 974 Abs. 2 ZGB;Art. 973 ZGB;...
2,train_0003,Das Kompetenzzentrum Völkerstrafrecht bei der ...,Art. 264m StGB,1095,1,Art. 264m StGB
3,train_0004,Die Linzer Stadtbahn AG ('LiSA') ist die priva...,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....,2899,4,Art. 176 Abs. 1 IPRG;Art. 186 Abs. 1 IPRG;Art....
4,train_0005,Die Stadt Winterthur beschloss am 10. Februar ...,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...,5763,6,Art. 93 Abs. 1 BGG;Art. 89 Abs. 1 BGG;Art. 89 ...


In [36]:
copy_train_data.describe()

,query_len,num_citations
count,1125.000000,1125.000000
mean,1622.479111,4.116444
std,3032.639461,4.525899
min,40.000000,1.000000
25%,442.000000,1.000000
50%,1016.000000,3.000000
75%,1794.000000,5.000000
max,32767.000000,44.000000


In [37]:
copy_train_data.to_csv(DATA_PATH / "processed" / "cleaned_train.csv", index=False)